# Clustering Whole Model

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load CNN Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


## Baseline model weight 확인 : 추후 clustered model의 weight와 비교 예정

In [ ]:
print(model.layers[0].get_weights()[0])

[[[[-1.51755229e-01  5.41861244e-02 -3.74114454e-01  6.90663308e-02
     2.30496298e-05 -1.55448973e-01  1.13818971e-02  2.24417031e-01
    -4.61155087e-01  3.23298909e-02  2.21437111e-01 -1.89202391e-02
     5.59891798e-02 -2.20387187e-02 -2.52020121e-01  1.88195724e-02
     1.53140306e-01  2.85692513e-01  5.25426209e-01 -5.55649817e-01
    -1.83962017e-01 -9.81873721e-02  1.49964496e-01 -4.72537994e-01
     1.84016690e-01 -2.12995529e-01  1.17162190e-01 -7.85708055e-02
    -1.95558578e-01  1.80093363e-01  1.01064458e-01  9.86170098e-02]]

  [[-2.72573411e-01  2.97346354e-01  1.66430905e-01 -8.50593392e-03
     1.53356224e-01  1.59565493e-01  9.12198573e-02  1.32788256e-01
    -8.64069164e-02  1.80115432e-01 -8.38115141e-02  1.25587016e-01
     5.66324815e-02 -1.34803681e-02 -9.77936462e-02 -6.16825372e-02
    -3.45552191e-02  2.63095707e-01 -5.19012511e-02 -1.73447430e-01
    -4.31877114e-02 -4.32206929e-01 -6.75449446e-02 -2.45501533e-01
     3.00595850e-01 -1.49667814e-01 -2.368108

## Clustering Whole Model
* number of cluserts : 16
* cluster centroids init : Linear ( Linear or KMean++ 추천 )

In [ ]:
clustering_params = {
  'number_of_clusters': 16,
  'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.LINEAR
}

clustered_model = tfmot.clustering.keras.cluster_weights(model,**clustering_params)

## Compile 후 model 확인 : clustering을 위해 metadata가 추가 된 model 확인

In [ ]:
clustered_model.compile(loss=keras.losses.SparseCategoricalCrossentropy(),
                        optimizer=keras.optimizers.Adam(learning_rate = 1e-5),  # 작은 leraning rate 사용
                        metrics=['accuracy'])

clustered_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 cluster_conv2d (ClusterWei  (None, 26, 26, 32)        624       
 ghts)                                                           
                                                                 
 cluster_max_pooling2d (Clu  (None, 13, 13, 32)        0         
 sterWeights)                                                    
                                                                 
 cluster_conv2d_1 (ClusterW  (None, 11, 11, 16)        9248      
 eights)                                                         
                                                                 
 cluster_max_pooling2d_1 (C  (None, 5, 5, 16)          0         
 lusterWeights)                                                  
                                                                 
 cluster_flatten (ClusterWe  (None, 400)               0

## Clustering을 위한 training

In [ ]:
hist_clustered = clustered_model.fit(
    train_images,
    train_labels,
    batch_size=500,
    epochs=1,
    validation_split=0.1
)

108/108 [==============================] - 5s 11ms/step - loss: 0.0058 - accuracy: 0.9979 - val_loss: 0.0455 - val_accuracy: 0.9903


## Accuracy 비교 : baseline model vs clustered model

In [ ]:
_, clustered_model_accuracy = clustered_model.evaluate(
  test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Clustered test accuracy:', clustered_model_accuracy)

Baseline test accuracy: 0.9904000163078308
Clustered test accuracy: 0.9914000034332275


## Clustered model의 weight 확인 : 16개의 cluster로 weight가 제한 됨

In [ ]:
final_model = tfmot.clustering.keras.strip_clustering(clustered_model)

print(final_model.layers[0].get_weights()[0])

[[[[-0.18485887  0.05181757 -0.3433365   0.05181757 -0.02689979
    -0.18485887 -0.02689979  0.21000396 -0.42203787  0.05181757
     0.21000396 -0.02689979  0.05181757 -0.02689979 -0.26391166
     0.05181757  0.13071126  0.2886438   0.5251123  -0.57974696
    -0.18485887 -0.10645057  0.13071126 -0.5012575   0.21000396
    -0.18485887  0.13071126 -0.10645057 -0.18485887  0.21000396
     0.13071126  0.13071126]]

  [[-0.26391166  0.2886438   0.13071126 -0.02689979  0.13071126
     0.13071126  0.05181757  0.13071126 -0.10645057  0.21000396
    -0.10645057  0.13071126  0.05181757 -0.02689979 -0.10645057
    -0.02689979 -0.02689979  0.2886438  -0.02689979 -0.18485887
    -0.02689979 -0.42203787 -0.10645057 -0.26391166  0.2886438
    -0.18485887 -0.26391166 -0.02689979 -0.3433365  -0.18485887
    -0.18485887 -0.02689979]]

  [[-0.02689979  0.36818716  0.13071126 -0.02689979  0.44610286
    -0.26391166  0.2886438   0.21000396  0.36818716  0.13071126
     0.05181757 -0.10645057 -0.10645057 -0.

In [ ]:
len(set(list(final_model.layers[0].get_weights()[0].reshape((-1)))))

16

## 최종 clustered model의 구조 확인
* baseline model과 같음 : memory size에서도 달라진 부분이 없음 (TFMOT clustering 의 한계)

In [ ]:
final_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

## LiteRT 모델로 변환 (Clustering whole model)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(final_model)
tflite_model = converter.convert()

In [ ]:
tflite_clustering_whole_model_file = save_dir + 'mnist_clustering_whole_model.tflite'
open(tflite_clustering_whole_model_file, 'wb').write(tflite_model)

233596

## 추론 속도 측정

* benchmark_model 설치

In [ ]:
!wget https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
!chmod +x linux_x86-64_benchmark_model

--2025-12-23 14:31:58--  https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
Resolving storage.googleapis.com (storage.googleapis.com)... 172.253.117.207, 142.250.99.207, 142.250.107.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.253.117.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6685264 (6.4M) [application/octet-stream]
Saving to: ‘linux_x86-64_benchmark_model’

linux_x86-64_benchm 100%[===================>]   6.38M  --.-KB/s    in 0.04s   

2025-12-23 14:31:58 (180 MB/s) - ‘linux_x86-64_benchmark_model’ saved [6685264/6685264]



* Baseline model

In [ ]:
tflite_baseline_model_file = save_dir + 'mnist_baseline_model.tflite'
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_baseline_model_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_baseline_model.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.233596
INFO: Initialized session in 4.996ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=11016 first=95 curr=43 min=37 max=359 avg=45.1179 std=9 p5=39 median=43 p95=61

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=21889 first=54 curr=43 min=33 max=192 avg=45.4155 std=8 p5=37 median=43 p95=64

INFO: Inference timings in us: Init: 4996, First inference: 95, Warmup (avg): 45.1179, Inference (avg)

* Clustering whole model

In [ ]:
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_clustering_whole_model_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_clustering_whole_model.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_clustering_whole_model.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_clustering_whole_model.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.233596
INFO: Initialized session in 5.369ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=11142 first=105 curr=43 min=33 max=672 avg=44.6054 std=13 p5=37 median=43 p95=67

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=22951 first=57 curr=43 min=33 max=200 avg=43.3192 std=7 p5=37 median=43 p95=60

INFO: Inference timings in us: Init: 5369, First inference: 105, Warmup (avg

## 모델 압축 테스트

* 압축 함수 정의

In [ ]:
import tempfile
import os
import zipfile

def get_zipped_model_size(model):

  _, model_file = tempfile.mkstemp('.h5')
  model.save(model_file)

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(model_file)

  return os.path.getsize(zipped_file)

* 압축된 파일 크기 비교

In [ ]:
size_zipped_baseline_model = get_zipped_model_size(model)
size_zipped_clustering_model = get_zipped_model_size(final_model)

print("Size of zipped baseline model file : {}".format(size_zipped_baseline_model))
print("Size of zipped clustering model file : {}".format(size_zipped_clustering_model))
print("ratio : {}".format(size_zipped_baseline_model/size_zipped_clustering_model))

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Size of zipped baseline model file : 421220
Size of zipped clustering model file : 35770
ratio : 11.77578976796198
